# Latihan 3 (Bonus): Server MCP Buatan Sendiri

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nur-Azzam-Wafiuddin/workshop/blob/uji-colab/colab/hari4/09-mcp-latihan-3.ipynb)

Ini versi **Google Colab**. Isi dan urutan contohnya sama dengan versi VS Code; yang berbeda hanya cell Persiapan di bawah, yang menyalin repository, memasang pustaka, dan mengisi API key di runtime Colab.

1. Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri.
2. Jalankan tiga cell **Persiapan Colab** berurutan (sekitar 2-3 menit pada sesi baru).
3. Kerjakan dari atas ke bawah dengan `Shift+Enter`.

Kode jawaban latihan ada di bagian **Kode Jawaban Latihan** pada [PDF bab ini](https://github.com/Nur-Azzam-Wafiuddin/workshop/blob/uji-colab/hari4/09-mcp/09-mcp.pdf). Salin kodenya ke cell yang bertanda `TODO`.

Kalau runtime terputus, file di `/content` (termasuk `.env`) hilang. Jalankan ulang Persiapan, lalu lanjutkan dari cell yang sedang Anda kerjakan.

## Persiapan Colab
Jalankan tiga cell berikut berurutan sebelum cell lain di notebook ini.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/Nur-Azzam-Wafiuddin/workshop.git"
REPO = "/content/workshop-genai-migas" if os.path.isdir("/content") else os.path.abspath("workshop-genai-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "uji-colab", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

Latihan ini berupa file Python, `hari4/09-mcp/latihan_3.py`. Di Colab file itu ditulis dari cell di bawah. Isi bagian bertanda `TODO(...)` dengan kode dari bagian **Kode Jawaban Latihan** pada PDF bab, lalu jalankan cell untuk menyimpan file. Jalankan ulang cell setiap kali kodenya diubah.

Baris pertama cell (`%%writefile ...`) menentukan nama file dan tidak boleh diubah.

In [ ]:
%%writefile hari4/09-mcp/latihan_3.py
"""
Server MCP buatan sendiri untuk Latihan 3 (Bonus): informasi work order.

Server stdio ini membuka dua tool hanya-baca:
- daftar_wo_terbuka(area)           : work order terbuka di sebuah area
- hitung_kerugian_downtime(...)     : perkiraan biaya downtime

Dipakai oleh latihan_4.py (klien yang menjalankannya sebagai subproses).
"""

import sys

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("work-order")

WORK_ORDER = {
    "Seruni": [
        ("WO-26-12501", "Kebocoran seal ESP SR-020", "tinggi"),
        ("WO-26-12502", "Lampu penerangan manifold berkedip", "rendah"),
    ],
    "Kenanga": [("WO-26-12611", "Sucker rod KN-033 patah, menunggu rig", "tinggi")],
    "Melati": [],
}


# TODO(B9.9): tulis tool daftar_wo_terbuka(area) dengan @mcp.tool(): cari WORK_ORDER menurut area (huruf besar di awal), raise ValueError yang menyebut area yang ada bila tidak dikenal, dan kembalikan satu baris per work order (id | deskripsi | prioritas)
pass  # TODO: ganti dengan implementasi Anda


# TODO(B9.10): tulis tool hitung_kerugian_downtime(downtime_jam, laju_bopd, harga_usd_per_bbl=70.0): raise ValueError bila negatif, hitung barel = laju_bopd * downtime_jam / 24, dan kembalikan teks 'Kehilangan X barel, perkiraan kerugian USD Y'
pass  # TODO: ganti dengan implementasi Anda


if __name__ == "__main__":
    print("Server MCP work order (stdio) berjalan...", file=sys.stderr)
    mcp.run(transport="stdio")